# ViHSD Mixture of Experts experiment

This Colab entry point mounts Google Drive, installs the project dependencies, and runs the root-level training and evaluation scripts. Change the experiment parameter cell before each run; there is no need to edit or push `configs/vihsd.yaml`.

## 1. Mount Google Drive

The YAML checkpoint path points to `/content/drive/MyDrive/ViHSD-MoE/checkpoints`. Drive must be mounted before training so `.safetensors` files persist after the Colab runtime ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the GitHub repository and install dependencies

This notebook treats GitHub as the source of truth. Each runtime clones the latest `main` branch into `/content/moe-vihsd`, installs dependencies from that clone, and runs the scripts there.

In [ ]:
PROJECT_DIR = '/content/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'

!rm -rf $PROJECT_DIR
!git clone --depth 1 --branch main $REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR
%pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Create a Colab Secret named HF_TOKEN before continuing.')
os.environ['HF_TOKEN'] = hf_token
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/checkpoints'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/results'
print('Hugging Face authentication configured from Colab Secrets.')

## 4. Set experiment parameters

Change only the values you want to test. These override the repository YAML for this run only, and the final configuration is saved with its checkpoint.

**How keys work:** use the same path as `configs/vihsd.yaml`, replacing YAML nesting with dots. For example, `model.num_experts` changes `model: -> num_experts:`. Any omitted key keeps its YAML default.

**Example:** to compare an 8-expert, top-2 router, set `model.num_experts` to `8` and `model.top_k` to `2`; use `RUN_ID = 'experts-8-topk-2'`. Keep `top_k` less than or equal to `num_experts`. Set `EXPERIMENT_OVERRIDES = {}` to use the untouched baseline.

In [ ]:
# Baseline: leave this empty to use every YAML default.
EXPERIMENT_OVERRIDES = {}

# Example experiment (uncomment this block to use it):
# EXPERIMENT_OVERRIDES = {
#     'training.epochs': 10,
#     'training.learning_rate': 0.0001,
#     'model.num_experts': 8,
#     'model.top_k': 2,
# }
RUN_ID = None  # e.g. 'experts-8-topk-2'; None creates a UTC timestamp
SMOKE_TEST = False

## 5. Train the full-parameter MoE

All embeddings, attention layers, router parameters, expert parameters, and classifier parameters are optimized. The best validation checkpoint and its resolved configuration are saved to Google Drive under a run folder.

In [ ]:
import json
import subprocess

command = ['python', 'train.py', '--config', 'configs/vihsd.yaml']
command.append('--smoke-test' if SMOKE_TEST else '--no-smoke-test')
if RUN_ID:
    command.extend(['--run-id', RUN_ID])
for key, value in EXPERIMENT_OVERRIDES.items():
    command.extend(['--set', f'{key}={json.dumps(value)}'])
print('Running:', ' '.join(command))
subprocess.run(command, check=True, env=os.environ.copy())

## 6. Evaluate the newest run and save JSON predictions

This loads the latest checkpoint from Drive, evaluates the test split, records expert routing counts, and writes results into the cloned repository.

In [ ]:
!CHECKPOINT_DIR=$CHECKPOINT_DIR RESULTS_DIR=$RESULTS_DIR python evaluate.py --config configs/vihsd.yaml